# 05: プレー分類モデルの定量評価

03 で学習した LSTM プレー分類モデルを、学習に使っていない検証動画で定量評価する。

## 評価設計

- **評価単位**: フレーム単位の2値分類（プレー中=1 / プレー外=0）
- **主指標**: Precision / Recall / F1（陽性率が約3割の不均衡データのため accuracy 単独では判断しない）
  - Precision が低い → プレーしていない区間が出力動画に混入する
  - Recall が低い → プレー区間を取りこぼす
- **補助指標**: 
  - Confusion Matrix
  - PR 曲線, Average Precision（閾値に依存しないモデル能力）
  - F1 を最大化する閾値
- **2段階で評価**（後処理の寄与を切り分ける）:
  1. **生モデル**:
    - LSTM が出した確率をフレームごとに単純平均しただけの値を、閾値 0.5 でそのまま 1/0 判定する。
    - モデル本体が生で持つ識別能力
  2. **後処理込み**:
  - 生モデルの確率系列に対し、本番推論（`PlaySceneDetector`）と同じ**メディアンフィルタ**（窓幅 `SMOOTHING_WINDOW` フレームで中央値を取り、単発の確率チラつきを均す平滑化処理）をかけてから閾値判定する。本番が実際に使っているのと同じ後処理
  - 両者は同一の生確率を入力に「メディアンフィルタの有無」だけを変えて計算しているため、指標の差分がそのまま後処理（平滑化）の寄与分になる
  - 差がほぼ無ければ後処理は省略可能、大きければ本番でも必須という判断材料になる
- **2スコープで評価**（予測が存在しないフレームの扱いで切り分ける）:
  1. **分類器単体**: 姿勢データが存在するフレームのみで評価。LSTM 分類器そのものの性能
  2. **製品出力（シーン区間）**: 本番 `PlaySceneDetector` と同じシーン抽出
     （閾値判定 → 最小シーン長フィルタ → 近接シーンマージ）で区間を作り、
     区間内の全フレームをプレー予測として正解ラベルの全フレームと比較。製品としての性能
  - 姿勢エクスポートは 15fps（元動画の 1/2）でフレームを間引くため、予測があるのは
    全ラベルフレームの約50%。間引かれたフレームもシーン区間内ならクリップされるので、
    製品評価は「予測なし=プレー外」ではなく区間展開で判定する
  - 両者の差 ≒ シーン化（最小長フィルタ・マージ）の影響＋骨格欠測区間の取りこぼし
- **集計**: 全フレームをプールしたマイクロ平均を主指標とし、動画別の内訳を併記する
  （フレームは時系列相関があり独立でないため、集計値1点では動画ごとの偏りが見えない）

## 注意

- 学習時の `best_val_f1` とは数値がずれる。学習時はシーケンスを flatten して計算する
  （stride=5 でも同一フレームが複数シーケンスに重複カウントされる）が、
  本評価はフレームごとに確率を集約してから計算するため。


In [ ]:
!pip install torch torchvision torchaudio
!pip install pandas numpy matplotlib tqdm scipy

In [ ]:
# Google Driveをマウント（Colab環境の場合）
import sys
import os
from pathlib import Path

try:
    import google.colab
    from google.colab import drive

    # Google Driveをマウント
    drive.mount("/content/drive")

    # プロジェクトディレクトリに移動
    PROJECT_ROOT = "/content/drive/MyDrive/Visuable_for_you_tabletennis"
    os.chdir(PROJECT_ROOT)

    # notebooksディレクトリをパスに追加
    sys.path.insert(0, os.path.join(PROJECT_ROOT, "scripts/notebooks"))

    print(f"✓ Google Driveをマウントしました")
    print(f"✓ プロジェクトルート: {PROJECT_ROOT}")
    IN_COLAB = True

except ImportError:
    # ローカル環境の場合
    IN_COLAB = False
    # notebooksディレクトリ（このノートブックの場所）をパスに追加
    notebook_dir = Path(__file__).parent if "__file__" in globals() else Path.cwd()
    sys.path.insert(0, str(notebook_dir))
    print(f"✓ ローカル環境で実行中")
    print(f"✓ 作業ディレクトリ: {Path.cwd()}")

# utilsをインポート
from utils import ColabFileManager

# ファイルマネージャーの初期化（プロジェクトルートは自動検出）
fm = ColabFileManager()

print(f"✓ ファイルマネージャー初期化完了")
print(f"  - 検出されたプロジェクトルート: {fm.project_root}")
print(f"  - Colab環境: {fm.is_colab}")

# ローカル環境ではプロジェクトルート（ml/）に移動し、相対パスと src import を揃える
if not IN_COLAB:
    os.chdir(fm.project_root)
    if str(fm.project_root) not in sys.path:
        sys.path.insert(0, str(fm.project_root))
    print(f"  - 作業ディレクトリをプロジェクトルートに変更: {Path.cwd()}")

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from scipy.ndimage import median_filter
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from src.datasets import CSVPoseSequenceDataset
from src.datasets.base_dataset import collate_fn
from src.models.play_classifier_lstm import PlayClassifierLSTM

print("インポート完了")
print(f"PyTorchバージョン: {torch.__version__}")
print(f"CUDAが利用可能: {torch.cuda.is_available()}")

In [ ]:
# ========================================
# 設定（ここを変更してください）
# ========================================

# 検証データ（03 の val_dirs と同一。学習には使っていない動画）
data_root = "data/detect"
val_dirs = [
    f"{data_root}/05_train",
    f"{data_root}/07_train",
    f"{data_root}/08_train",
    f"{data_root}/12_train",
    f"{data_root}/14_train",
]

# 評価対象モデル
# 学習出力を直接評価する場合は output/training/<timestamp>/best_model.pth と config.json を指定
MODEL_PATH = "models/play_classifier/lstm_model.pth"
MODEL_CONFIG_PATH = "models/play_classifier/ltsm_config.json"

# 後処理パラメータ（本番 PlaySceneDetectionConfig / _extract_scenes のデフォルトに合わせる）
THRESHOLD = 0.5  # プレー判定の閾値
SMOOTHING_WINDOW = 5  # メディアンフィルタのウィンドウサイズ（0/1 で無効）
MIN_SCENE_DURATION = 10  # 最小シーン長（フレーム数）
MERGE_GAP = 15  # この間隔（フレーム数）以内の隣接シーンをマージ

# 出力ディレクトリ
OUTPUT_DIR = Path(f"output/evaluation/{datetime.now().strftime('%Y%m%d_%H%M%S')}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ファイル存在確認
print("=" * 60)
print("ファイルの確認")
print("=" * 60)
print(f"モデル: {'✓' if os.path.exists(MODEL_PATH) else '✗'} {MODEL_PATH}")
print(
    f"モデル設定: {'✓' if os.path.exists(MODEL_CONFIG_PATH) else '✗'} {MODEL_CONFIG_PATH}"
)
print(f"\n検証用データ:")
for i, data_dir in enumerate(val_dirs, 1):
    csv_exists = os.path.exists(os.path.join(data_dir, "player_pose_data.csv"))
    label_exists = os.path.exists(os.path.join(data_dir, "play_labels.csv"))
    print(
        f"  [{i}] {os.path.basename(data_dir)}: "
        f"pose {'✓' if csv_exists else '✗'} / labels {'✓' if label_exists else '✗'}"
    )
print(f"\n出力先: {OUTPUT_DIR}")

In [ ]:
# ========================================
# モデルの読み込み
# ========================================

with open(MODEL_CONFIG_PATH, "r") as f:
    config_dict = json.load(f)

# 学習出力の config.json（model/dataset 階層あり）とフラット形式の両方に対応
if "model" in config_dict:
    model_cfg = config_dict["model"]
    sequence_length = config_dict.get("dataset", {}).get("sequence_length", 30)
else:
    model_cfg = config_dict
    sequence_length = config_dict.get("sequence_length", 30)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PlayClassifierLSTM(
    input_size=102,  # 座標34 + 速度34 + 加速度34
    hidden_size=model_cfg.get("hidden_size", 128),
    num_layers=model_cfg.get("num_layers", 2),
    dropout=model_cfg.get("dropout", 0.3),
)

checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"エポック: {checkpoint.get('epoch', 'N/A')}")
    print(f"学習時 Best Val F1: {checkpoint.get('best_val_f1', 'N/A')}")
else:
    model.load_state_dict(checkpoint)

model = model.to(device)
model.eval()

print(f"\nモデルを読み込み: {MODEL_PATH}")
print(f"  デバイス: {device}")
print(f"  シーケンス長: {sequence_length}")

In [ ]:
# ========================================
# フレーム単位の予測確率を算出
# ========================================
# 本番推論（PlaySceneDetector._predict）と同じ方式:
#   stride=1 のオーバーラップ窓で推論し、フレームごとに確率を平均
#   → メディアンフィルタでスムージング → 閾値判定
# 相違点: フレーム番号は metadata の start_idx/end_idx から dataset.frames を
#   引いて正確に対応づける（複数トラックの行やフレーム欠落があっても壊れない）


def predict_frame_probs(data_dir):
    """1動画分のフレーム単位確率を算出し、正解ラベルと突き合わせる"""
    csv_path = Path(data_dir) / "player_pose_data.csv"
    label_path = Path(data_dir) / "play_labels.csv"

    dataset = CSVPoseSequenceDataset(
        csv_path=str(csv_path),
        label_path=str(label_path),
        sequence_length=sequence_length,
        stride=1,
        keypoint_features=None,
    )
    loader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
    )

    frame_probs = {}  # {frame番号: [確率, ...]}
    with torch.no_grad():
        for features, _, metadata in tqdm(loader, desc=Path(data_dir).name):
            logits = model(features.to(device))  # (batch, seq, 1)
            probs = torch.sigmoid(logits).squeeze(-1).cpu().numpy()
            probs = probs.reshape(
                len(metadata), -1
            )  # batch=1 でも (batch, seq) に揃える
            for row, md in zip(probs, metadata):
                seq_frames = dataset.frames[md["start_idx"] : md["end_idx"]]
                for f, p in zip(seq_frames, row):
                    frame_probs.setdefault(int(f), []).append(float(p))

    # フレームごとに確率を平均（オーバーラップ窓・複数トラックをプール）
    pred_frames = np.array(sorted(frame_probs.keys()))
    raw_probs = np.array([np.mean(frame_probs[f]) for f in pred_frames])

    # メディアンフィルタ（予測が存在するフレーム系列上で適用: 本番と同じ）
    smoothed_probs = raw_probs.copy()
    if SMOOTHING_WINDOW > 1 and len(smoothed_probs) > SMOOTHING_WINDOW:
        smoothed_probs = median_filter(smoothed_probs, size=SMOOTHING_WINDOW)

    # 正解ラベル（全フレーム）と対応づけ。予測のないフレームは確率 0.0（プレー外扱い）
    labels_df = pd.read_csv(label_path)
    all_frames = labels_df["frame"].values.astype(int)
    y_true = labels_df["label"].values.astype(int)

    frame_to_idx = {f: i for i, f in enumerate(all_frames)}
    raw_full = np.zeros(len(all_frames))
    smoothed_full = np.zeros(len(all_frames))
    covered = np.zeros(
        len(all_frames), dtype=bool
    )  # 骨格が検出され予測が存在するフレーム
    for f, rp, sp in zip(pred_frames, raw_probs, smoothed_probs):
        idx = frame_to_idx.get(f)
        if idx is not None:
            raw_full[idx] = rp
            smoothed_full[idx] = sp
            covered[idx] = True

    return {
        "name": Path(data_dir).name,
        "frames": all_frames,
        "y_true": y_true,
        "raw_probs": raw_full,
        "smoothed_probs": smoothed_full,
        "covered": covered,
        "coverage": covered.mean(),
    }


results = [predict_frame_probs(d) for d in val_dirs]

print("\nフレーム単位確率の算出完了:")
for r in results:
    print(
        f"  {r['name']}: {len(r['y_true'])}フレーム, "
        f"姿勢データあり {r['coverage'] * 100:.1f}%"
    )
print(
    "※ 姿勢エクスポートが 15fps（元動画の 1/2）でフレームを間引くため、約50%が正常値。"
)
print("   それを大きく下回る場合は上流（姿勢推定・トラッキング）の検出漏れを疑う。")

In [ ]:
# ========================================
# 【分類器単体】Precision / Recall / F1 と Confusion Matrix
# ========================================
# 姿勢データが存在するフレームのみで評価（LSTM 分類器そのものの性能）


def compute_metrics(y_true, y_pred):
    """2値分類の評価指標を計算（training_pipeline._compute_metrics と同じ定義）"""
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def finalize_table(rows):
    """動画別の行に全体（マイクロ）とマクロ平均を追加したテーブルを作成"""
    df = pd.DataFrame(rows)

    # マイクロ平均: 全動画の TP/FP/FN/TN を合算してから指標を計算（主指標）
    tp, fp, fn, tn = (
        int(df["tp"].sum()),
        int(df["fp"].sum()),
        int(df["fn"].sum()),
        int(df["tn"].sum()),
    )
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )
    micro = {
        "video": "全体 (micro)",
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "accuracy": (tp + tn) / (tp + fp + fn + tn),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

    # マクロ平均: 動画ごとの指標の単純平均（動画間のばらつき確認用）
    macro = {
        "video": "マクロ平均",
        "tp": None,
        "fp": None,
        "fn": None,
        "tn": None,
        "accuracy": df["accuracy"].mean(),
        "precision": df["precision"].mean(),
        "recall": df["recall"].mean(),
        "f1": df["f1"].mean(),
    }

    return pd.concat([df, pd.DataFrame([micro, macro])], ignore_index=True)


def classifier_metrics_table(prob_key, threshold):
    """姿勢データが存在するフレームのみの指標テーブル

    Args:
        prob_key: 'raw_probs'（生モデル）か 'smoothed_probs'（後処理込み）
        threshold: プレー判定の閾値
    """
    rows = []
    for r in results:
        mask = r["covered"]
        m = compute_metrics(
            r["y_true"][mask], (r[prob_key][mask] >= threshold).astype(int)
        )
        rows.append({"video": r["name"], **m})
    return finalize_table(rows)


clf_raw_table = classifier_metrics_table("raw_probs", 0.5)
clf_smoothed_table = classifier_metrics_table("smoothed_probs", THRESHOLD)

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print("=" * 70)
print("【分類器単体】生モデル（確率平均のみ, 閾値 0.5）")
print("=" * 70)
display(clf_raw_table)
print("=" * 70)
print(
    f"【分類器単体】後処理込み（メディアンフィルタ window={SMOOTHING_WINDOW}, 閾値 {THRESHOLD}）"
)
print("=" * 70)
display(clf_smoothed_table)

In [ ]:
# ========================================
# 【製品出力】シーン区間に展開したフレーム評価
# ========================================
# 本番 PlaySceneDetector._extract_scenes / _merge_close_scenes と同じロジックで
# シーン区間を抽出し、区間内の全フレームをプレー予測として正解ラベルと比較する。
# fps 間引きで予測がないフレームも、シーン区間内であればクリップに含まれるため
# プレー予測として扱う（製品挙動と一致）。


def extract_scenes(frames, preds, min_scene_duration, merge_gap):
    """連続したプレー区間を抽出し、近接シーンをマージ（本番と同じロジック）"""
    scenes = []
    in_scene = False
    scene_start = None

    for frame, pred in zip(frames, preds):
        if pred == 1:  # プレー中
            if not in_scene:
                scene_start = frame
                in_scene = True
        else:  # プレー外
            if in_scene:
                if frame - scene_start >= min_scene_duration:
                    scenes.append((scene_start, frame - 1))
                in_scene = False
    if in_scene and frames[-1] - scene_start >= min_scene_duration:
        scenes.append((scene_start, frames[-1]))

    # 間隔が merge_gap 以内の隣接シーンをマージ
    if len(scenes) > 1:
        merged = [scenes[0]]
        for start, end in scenes[1:]:
            prev_start, prev_end = merged[-1]
            if start - prev_end <= merge_gap:
                merged[-1] = (prev_start, end)
            else:
                merged.append((start, end))
        scenes = merged
    return scenes


rows = []
for r in results:
    mask = r["covered"]
    sampled_frames = r["frames"][mask]
    sampled_preds = (r["smoothed_probs"][mask] >= THRESHOLD).astype(int)
    scenes = extract_scenes(
        sampled_frames, sampled_preds, MIN_SCENE_DURATION, MERGE_GAP
    )

    # シーン区間内の全ラベルフレームをプレー予測(1)とする
    scene_pred = np.zeros(len(r["frames"]), dtype=int)
    for start, end in scenes:
        scene_pred[(r["frames"] >= start) & (r["frames"] <= end)] = 1

    r["scenes"] = scenes
    r["scene_pred"] = scene_pred

    m = compute_metrics(r["y_true"], scene_pred)
    rows.append({"video": r["name"], "num_scenes": len(scenes), **m})

product_table = finalize_table(rows)

print("=" * 70)
print(
    f"【製品出力】シーン区間展開後・全フレーム "
    f"(min_scene_duration={MIN_SCENE_DURATION}, merge_gap={MERGE_GAP})"
)
print("=" * 70)
display(product_table)
print("※ 分類器単体との差は、シーン化（最小シーン長・近接マージ）の影響と")
print("   骨格欠測区間（姿勢が取れずシーンにならない部分）の取りこぼし分。")

In [ ]:
# ========================================
# PR 曲線・Average Precision・閾値の較正
# ========================================
# 分類器単体のスコープ（姿勢データが存在するフレームのみ）を全動画プールして計算。
# 予測が存在しないフレーム（fps 間引き・骨格未検出）は確率が定義されないため除外する。

covered_all = np.concatenate([r["covered"] for r in results])
y_true_all = np.concatenate([r["y_true"] for r in results])[covered_all]
raw_all = np.concatenate([r["raw_probs"] for r in results])[covered_all]
smoothed_all = np.concatenate([r["smoothed_probs"] for r in results])[covered_all]


def pr_curve(y_true, probs):
    """PR 曲線と Average Precision を計算（確率の降順に閾値を動かす）"""
    order = np.argsort(-probs, kind="stable")
    y = y_true[order]
    tp = np.cumsum(y)
    fp = np.cumsum(1 - y)
    precision = tp / (tp + fp)
    recall = tp / y_true.sum()
    # AP = Σ (recall の増分 × その点の precision)
    ap = float(np.sum(np.diff(np.concatenate([[0.0], recall])) * precision))
    return precision, recall, ap


prec_raw, rec_raw, ap_raw = pr_curve(y_true_all, raw_all)
prec_sm, rec_sm, ap_sm = pr_curve(y_true_all, smoothed_all)

# F1 を最大化する閾値を検証データで探索（本番 threshold 設定の較正に使う）
thresholds = np.arange(0.05, 1.00, 0.05)
f1_raw_list = [
    compute_metrics(y_true_all, (raw_all >= t).astype(int))["f1"] for t in thresholds
]
f1_sm_list = [
    compute_metrics(y_true_all, (smoothed_all >= t).astype(int))["f1"]
    for t in thresholds
]
best_threshold = float(thresholds[int(np.argmax(f1_sm_list))])
best_f1 = float(np.max(f1_sm_list))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR曲線
axes[0].plot(rec_raw, prec_raw, label=f"Raw (AP={ap_raw:.4f})", linewidth=2)
axes[0].plot(rec_sm, prec_sm, label=f"Smoothed (AP={ap_sm:.4f})", linewidth=2)
baseline = float(y_true_all.mean())
axes[0].axhline(
    baseline,
    color="gray",
    linestyle="--",
    linewidth=1,
    label=f"Chance ({baseline:.3f})",
)
axes[0].set_xlabel("Recall", fontsize=12)
axes[0].set_ylabel("Precision", fontsize=12)
axes[0].set_title("Precision-Recall Curve (covered frames)", fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# F1 vs 閾値
axes[1].plot(thresholds, f1_raw_list, marker="o", label="Raw")
axes[1].plot(thresholds, f1_sm_list, marker="o", label="Smoothed")
axes[1].axvline(
    best_threshold,
    color="red",
    linestyle="--",
    linewidth=1,
    label=f"Best threshold {best_threshold:.2f} (F1={best_f1:.4f})",
)
axes[1].set_xlabel("Threshold", fontsize=12)
axes[1].set_ylabel("F1 Score", fontsize=12)
axes[1].set_title("F1 Score vs Threshold (covered frames)", fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pr_curve_and_threshold.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Average Precision (Raw):      {ap_raw:.4f}")
print(f"Average Precision (Smoothed): {ap_sm:.4f}")
print(f"F1 最大化閾値（Smoothed, 分類器単体）: {best_threshold:.2f} (F1={best_f1:.4f})")
print("※ 本番 PlaySceneDetectionConfig.threshold の較正値として利用できる")

In [ ]:
# ========================================
# 動画ごとの予測タイムライン（定性確認）
# ========================================
# 緑帯 = 正解プレー区間 / 橙帯 = 検出シーン（製品出力）/ 青線 = 予測確率（後処理込み）/ 赤破線 = 閾値
# 取りこぼし（緑帯だけの区間）と誤検出（橙帯だけの区間）が目視できる

fig, axes = plt.subplots(len(results), 1, figsize=(16, 2.5 * len(results)))
if len(results) == 1:
    axes = [axes]

for ax, r in zip(axes, results):
    frames = r["frames"]
    mask = r["covered"]
    ax.fill_between(
        frames, 0, 1, where=r["y_true"] == 1, color="green", alpha=0.2, label="GT play"
    )
    ax.fill_between(
        frames,
        0,
        1,
        where=r["scene_pred"] == 1,
        color="orange",
        alpha=0.25,
        label="Detected scene",
    )
    # 確率線は姿勢データが存在するフレームのみ描画（間引きフレームを0で結ぶと歪むため）
    ax.plot(
        frames[mask],
        r["smoothed_probs"][mask],
        linewidth=0.8,
        label="Predicted prob (smoothed)",
    )
    ax.axhline(
        THRESHOLD,
        color="red",
        linestyle="--",
        linewidth=1,
        label=f"Threshold {THRESHOLD}",
    )
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel("Probability", fontsize=10)
    ax.set_title(r["name"], fontsize=12)
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Frame", fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "prediction_timelines.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ========================================
# 結果の保存
# ========================================

# 指標テーブル
clf_raw_table.to_csv(OUTPUT_DIR / "metrics_classifier_raw.csv", index=False)
clf_smoothed_table.to_csv(OUTPUT_DIR / "metrics_classifier_smoothed.csv", index=False)
product_table.to_csv(OUTPUT_DIR / "metrics_product.csv", index=False)

# 検出シーン区間（製品出力）
scenes_df = pd.concat(
    [
        pd.DataFrame(r["scenes"], columns=["start_frame", "end_frame"]).assign(
            video=r["name"]
        )
        for r in results
    ],
    ignore_index=True,
)[["video", "start_frame", "end_frame"]]
scenes_df.to_csv(OUTPUT_DIR / "detected_scenes.csv", index=False)

# フレーム単位の予測（後で区間レベル評価などに再利用できる）
pred_df = pd.concat(
    [
        pd.DataFrame(
            {
                "video": r["name"],
                "frame": r["frames"],
                "label": r["y_true"],
                "covered": r["covered"].astype(int),
                "raw_prob": r["raw_probs"],
                "smoothed_prob": r["smoothed_probs"],
                "scene_pred": r["scene_pred"],
            }
        )
        for r in results
    ],
    ignore_index=True,
)
pred_df.to_csv(OUTPUT_DIR / "frame_predictions.csv", index=False)


# サマリ JSON
def micro_row(table):
    row = table[table["video"] == "全体 (micro)"].iloc[0]
    return {k: float(row[k]) for k in ("precision", "recall", "f1", "accuracy")}


summary = {
    "evaluated_at": datetime.now().isoformat(),
    "model_path": str(MODEL_PATH),
    "model_config_path": str(MODEL_CONFIG_PATH),
    "val_dirs": val_dirs,
    "sequence_length": sequence_length,
    "threshold": THRESHOLD,
    "smoothing_window": SMOOTHING_WINDOW,
    "min_scene_duration": MIN_SCENE_DURATION,
    "merge_gap": MERGE_GAP,
    "classifier_raw_micro": micro_row(clf_raw_table),
    "classifier_smoothed_micro": micro_row(clf_smoothed_table),
    "product_scene_micro": micro_row(product_table),
    "average_precision_raw": ap_raw,
    "average_precision_smoothed": ap_sm,
    "best_threshold_smoothed": best_threshold,
    "best_f1_smoothed": best_f1,
    "coverage": {r["name"]: float(r["coverage"]) for r in results},
}
with open(OUTPUT_DIR / "summary.json", "w") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"評価結果を保存: {OUTPUT_DIR}")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  - {p.name}")

In [ ]:
# ========================================
# Google Driveに保存（Colab環境の場合）
# ========================================

import shutil

if IN_COLAB:
    SAVE_TO_DRIVE = (
        f"/content/drive/MyDrive/evaluation_results/play_classifier/{OUTPUT_DIR.name}"
    )

    os.makedirs(SAVE_TO_DRIVE, exist_ok=True)

    for src in OUTPUT_DIR.iterdir():
        dst = os.path.join(SAVE_TO_DRIVE, src.name)
        shutil.copy2(src, dst)
        print(f"コピー完了: {src.name} -> {dst}")

    print(f"\n評価結果をGoogle Driveに保存: {SAVE_TO_DRIVE}")
else:
    print("ローカル環境では自動保存はスキップされます")
    print(f"出力ディレクトリ: {OUTPUT_DIR}")